# CS781 Phase 2 — A3 LoRA (improved)

This is the same Phase 1 Architecture 3 and tokenizer. I changed the Phase 2 NER fine-tuning and decoding:

- Longer overlapping windows (256 tokens) and up to 30 training epochs, with early stopping after 5 epochs without a better validation F1.
- Separate learning rates for LoRA and the new classification head.
- Less aggressive down-weighting of the `O` class.
- Merge window logits **per token before** converting BIO tags into character spans.
- Compare greedy and valid-BIO Viterbi decoding on the same validation documents.

The training split and seed are unchanged. This run uses a new checkpoint folder, so the old `a3_lora` and `a3_lora_v2` runs are not overwritten. These are experiments, not guaranteed score improvements.


In [1]:
# Libraries needed for the Phase 2 notebooks.
# If torchao is already imported, restart Kaggle after running this cell.
%pip -q install "transformers>=4.45,<5" huggingface_hub safetensors tokenizers peft tqdm

# Kaggle may have torchao 0.10.0, which crashes with newer PEFT.
# Standard LoRA here does not need torchao.
%pip -q uninstall -y torchao


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:

# Phase 2 - a3_lora
# Edit HF_REPO_NAME below when your Phase 1 repo has a different name.
import os, json, random, math, time, zipfile, re
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizerFast
from huggingface_hub import HfApi, hf_hub_download

SEED = 44
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ARCH = 'a3'
TECHNIQUE = 'lora'
EXP_ID = 'a3_lora_v3_30ep'  # Naya folder, v2 checkpoint overwrite nahi hoga
# edit ONLY the name after your username, not your username itself
HF_REPO_NAME = 'cs781-phase1-architecture3'  # CHANGE THIS if the name is different on your Hugging Face account
# if tokenizer.json is not in the model repo, put a tokenizer repo name here
TOKENIZER_REPO_NAME = ''     # CHANGE THIS only if the checkpoint repo has no tokenizer
# alternatively, a Kaggle input directory that contains tokenizer.json
LOCAL_TOKENIZER_DIR = ''    # CHANGE THIS only if needed

OUTPUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUTPUT = OUTPUT / ('phase2_' + EXP_ID)
OUTPUT.mkdir(parents=True,exist_ok=True)

DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
MAX_LEN = 256     # Thoda more context for each window
STRIDE = 192      # 64-token overlap, so boundary entities get another chance
EPOCHS = 30      # Maximum epochs; early stopping may finish sooner
PATIENCE = 5     # Agar 5 epochs tak val F1 improve na ho, stop
LR_LORA = 1e-4   # Only the small LoRA matrices
LR_HEAD = 2e-4   # New NER classifier ko thoda higher LR
BATCH_SIZE = 1    # avoids padding. It also works with architecture 2's encoder.
ACCUM_STEPS = 4   # update weights every 4 pieces
O_WEIGHT = 0.5   # 0.2 pe kaafi extra entities predict ho rahi thi
VAL_PART = 0.10
MAX_TRAIN_DOCS = None  # All 63 available training docs; 100 was not a real cap
MAX_VAL_DOCS = None  # Same held-out split, evaluate all 7 docs
DECODE_METHOD = 'viterbi'  # Used during training validation
AUTO_SELECT_DECODER = True  # At the end, choose decoding by held-out F1
RANK = 8
ALPHA = 16
KD_TEMP = 2.0
KD_MIX = 0.5
# Knowledge distillation ONLY: location of trained A3 LoRA teacher
TEACHER_FILE = '/kaggle/working/phase2_a3_lora/a3_lora_best.pt'  # CHANGE only for KD
TEACHER_REPO_NAME = ''  # KD only: A3 teacher HF repo name, when working in a different session

# Same secret name used in your Phase 1 notebooks: HF
HF_TOKEN = os.environ.get('HF_TOKEN')
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF') or HF_TOKEN
except Exception:
    pass
api = HfApi(token=HF_TOKEN)
HF_USER = api.whoami()['name']      # read the active HF account, no hard-coded username
HF_REPO = HF_USER + '/' + HF_REPO_NAME
print('Logged in as:', HF_USER)
print('Loading Phase 1 from:', HF_REPO)
print('Experiment:', EXP_ID, TECHNIQUE)


Logged in as: P-261110044
Loading Phase 1 from: P-261110044/cs781-phase1-architecture3
Experiment: a3_lora_v3_30ep lora


In [3]:
# Exact architecture functions and model classes from original Phase 1 notebook.
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.modeling_outputs import CausalLMOutput


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Compute the RMS in fp32 for numerical stability, then cast back.
        rms = x.float().pow(2).mean(dim=-1, keepdim=True)
        x_norm = x * torch.rsqrt(rms + self.eps).to(x.dtype)
        return x_norm * self.weight

def rotate_half(x):
    # RoPE rotates each consecutive pair: (x0, x1) -> (-x1, x0).
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE needs an even head dimension."
        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)
        )
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def cos_sin(self, positions, dtype):
        # positions: [seq]
        freqs = torch.outer(positions.float(), self.inv_freq)
        cos = freqs.cos().repeat_interleave(2, dim=-1).to(dtype)
        sin = freqs.sin().repeat_interleave(2, dim=-1).to(dtype)
        return cos[None, None, :, :], sin[None, None, :, :]

    def apply_rotary(self, q, k, positions):
        cos, sin = self.cos_sin(positions, q.dtype)
        q = (q * cos) + (rotate_half(q) * sin)
        k = (k * cos) + (rotate_half(k) * sin)
        return q, k

class GroupedQueryLocalAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        window_size,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        assert dim % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.groups = num_heads // num_kv_heads
        self.window_size = window_size
        self.chunk_size = chunk_size
        self.dropout = dropout

        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)

        self.rope = RotaryEmbedding(self.head_dim)

    def _repeat_kv(self, x):
        # [B, kv_heads, T, D] -> [B, query_heads, T, D]
        return x.repeat_interleave(self.groups, dim=1)

    def _local_attention(self, q, k, v):
        """
        Exact causal sliding-window attention.

        We process a few query positions at a time.  Each query chunk only
        materializes the K/V region that can actually be attended to, instead
        of creating a full 2048 x 2048 attention matrix.
        """
        B, H, T, D = q.shape
        outputs = []

        k_rep = self._repeat_kv(k)
        v_rep = self._repeat_kv(v)

        for q_start in range(0, T, self.chunk_size):
            q_end = min(q_start + self.chunk_size, T)

            # Earliest key any query in this chunk could legally use.
            k_start = max(0, q_start - self.window_size + 1)
            k_end = q_end

            q_chunk = q[:, :, q_start:q_end, :]
            k_chunk = k_rep[:, :, k_start:k_end, :]
            v_chunk = v_rep[:, :, k_start:k_end, :]

            q_pos = torch.arange(q_start, q_end, device=q.device)
            k_pos = torch.arange(k_start, k_end, device=q.device)

            # True = allowed attention position.
            causal = k_pos[None, :] <= q_pos[:, None]
            within_window = k_pos[None, :] >= (
                q_pos[:, None] - self.window_size + 1
            )
            attn_mask = (causal & within_window)[None, None, :, :]

            out = F.scaled_dot_product_attention(
                q_chunk,
                k_chunk,
                v_chunk,
                attn_mask=attn_mask,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=False,
            )
            outputs.append(out)

        return torch.cat(outputs, dim=2)

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T, _ = x.shape

        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=x.device,
        )
        q, k = self.rope.apply_rotary(q, k, positions)

        if past_kv is None:
            # Training/prefill path: causal local attention over this sequence.
            attn_out = self._local_attention(q, k, v)
            k_all, v_all = k, v
        else:
            # Generation path.  We decode one token at a time after prefill.
            assert T == 1, "Cached decoding expects one new token at a time."
            past_k, past_v = past_kv
            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)

            # Keep only the most recent local window in the KV cache.
            k_all = k_all[:, :, -self.window_size:, :]
            v_all = v_all[:, :, -self.window_size:, :]

            k_rep = self._repeat_kv(k_all)
            v_rep = self._repeat_kv(v_all)

            # Every cached token is in the past (or the current token), so no
            # extra causal mask is necessary for this one-token decode step.
            attn_out = F.scaled_dot_product_attention(
                q,
                k_rep,
                v_rep,
                dropout_p=0.0,
                is_causal=False,
            )

        new_kv = None
        if use_cache:
            new_kv = (
                k_all[:, :, -self.window_size:, :],
                v_all[:, :, -self.window_size:, :],
            )

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.dim)
        return self.o_proj(attn_out), new_kv

class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))

class ModernTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        ffn_dim,
        window_size,
        chunk_size,
        dropout,
    ):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = GroupedQueryLocalAttention(
            dim=dim,
            num_heads=num_heads,
            num_kv_heads=num_kv_heads,
            window_size=window_size,
            chunk_size=chunk_size,
            dropout=dropout,
        )
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, ffn_dim, dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, position_offset=0):
        attn_out, new_kv = self.attn(
            self.attn_norm(x),
            past_kv=past_kv,
            use_cache=use_cache,
            position_offset=position_offset,
        )
        x = x + self.resid_dropout(attn_out)
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv

class ModernLocalTransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim=600,
        num_layers=8,
        num_heads=10,
        num_kv_heads=5,
        ffn_dim=1600,
        window_size=512,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim
        self.num_layers = num_layers

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.embed_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            ModernTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                num_kv_heads=num_kv_heads,
                ffn_dim=ffn_dim,
                window_size=window_size,
                chunk_size=chunk_size,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Weight tying saves parameters and is standard for causal LMs.
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_offset=0,
    ):
        x = self.embed_dropout(self.token_embedding(input_ids))
        new_caches = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, new_kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                position_offset=position_offset,
            )
            if use_cache:
                new_caches.append(new_kv)

        x = self.final_norm(x)
        return x, new_caches

    def forward(self, input_ids, labels=None, attention_mask=None, **kwargs):
        x, _ = self._run_hidden(input_ids)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            # Position t predicts token t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(loss=loss, logits=logits)

    @torch.inference_mode()
    def prefill(self, input_ids):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=None,
            use_cache=True,
            position_offset=0,
        )
        return self.lm_head(x), caches

    @torch.inference_mode()
    def decode_step(self, input_ids, past_key_values, position_offset):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            position_offset=position_offset,
        )
        return self.lm_head(x[:, -1, :]), caches

# Read the saved Phase 1 model and tokenizer from my Hugging Face repo.
# It may contain final_model/ or checkpoint-6104/ (or some other checkpoint).
import shutil
repo_files = api.list_repo_files(HF_REPO, repo_type='model')
print('Phase 1 repo:', HF_REPO)
print('Files in repo:', repo_files[:12])

weight_names = ('model_state.pt', 'model.safetensors', 'pytorch_model.bin')
final_folders = ('architecture-3-final/', 'architecture-4-final/',
                 'final_model/', 'legal-gpt2-final/', 'custom-decoder-final/', '')
checkpoint_folders = sorted(
    {f.split('/')[0] + '/' for f in repo_files if re.match(r'checkpoint-\d+/', f)},
    key=lambda folder: int(folder.split('-')[1].strip('/')),
    reverse=True
)

model_file = None
model_folder = None
for folder in list(final_folders) + checkpoint_folders:
    for name in weight_names:
        if folder + name in repo_files:
            model_file = folder + name
            model_folder = folder
            break
    if model_file is not None:
        break
if model_file is None:
    raise FileNotFoundError(
        'No model weights found. The repo must contain extracted model_state.pt '
        'or model.safetensors, not only a backup ZIP. Files: ' + str(repo_files[:30]))
print('Using weights:', model_file)

P1_DIR = OUTPUT / 'phase1_files'
# Clear files from an earlier run so we never mix two tokenizers or checkpoints.
if P1_DIR.exists():
    shutil.rmtree(P1_DIR)
P1_DIR.mkdir(parents=True)

def get_file(repo_id, repo_path):
    cached = hf_hub_download(repo_id=repo_id, filename=repo_path,
                             repo_type='model', token=HF_TOKEN)
    destination = P1_DIR / Path(repo_path).name
    shutil.copyfile(cached, destination)
    return destination

weights_path = get_file(HF_REPO, model_file)
weight_file = Path(model_file).name

# Architecture 3 has its config at the repo root, outside checkpoint-6104.
for config_file in (model_folder + 'architecture_config.json',
                    'architecture_config.json'):
    if config_file in repo_files:
        get_file(HF_REPO, config_file)
        break

# Use the same tokenizer that was used to pretrain this model.
tokenizer_folders = [model_folder, '', 'final_model/',
                     'architecture-3-final/', 'architecture-4-final/']
tokenizer_folders.extend(checkpoint_folders)
tok_folder = next((folder for folder in tokenizer_folders
                   if folder + 'tokenizer.json' in repo_files), None)
tokenizer_files = ('tokenizer.json', 'tokenizer_config.json',
                   'special_tokens_map.json', 'added_tokens.json')

if tok_folder is not None:
    for name in tokenizer_files:
        if tok_folder + name in repo_files:
            get_file(HF_REPO, tok_folder + name)
elif LOCAL_TOKENIZER_DIR:
    for name in tokenizer_files:
        source = Path(LOCAL_TOKENIZER_DIR) / name
        if source.exists():
            shutil.copyfile(source, P1_DIR / name)
elif TOKENIZER_REPO_NAME:
    other_repo = HF_USER + '/' + TOKENIZER_REPO_NAME
    other_files = api.list_repo_files(other_repo, repo_type='model')
    other_folder = next((x for x in ('', 'final_model/')
                         if x + 'tokenizer.json' in other_files), None)
    if other_folder is None:
        raise FileNotFoundError('Tokenizer missing from ' + other_repo)
    for name in tokenizer_files:
        if other_folder + name in other_files:
            get_file(other_repo, other_folder + name)

if not (P1_DIR / 'tokenizer.json').is_file():
    raise FileNotFoundError('Tokenizer missing. Upload tokenizer.json or set LOCAL_TOKENIZER_DIR.')

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    str(P1_DIR), pad_token='<pad>', unk_token='<unk>',
    bos_token='<s>', eos_token='</s>'
)
if not tokenizer.is_fast:
    raise RuntimeError('A fast tokenizer is required for character offsets.')

config_path = P1_DIR / 'architecture_config.json'
saved_cfg = json.loads(config_path.read_text()) if config_path.is_file() else {}
if 'vocab_size' in saved_cfg and len(tokenizer) != saved_cfg['vocab_size']:
    raise ValueError('Phase 1 vocabulary and tokenizer sizes differ.')

# These are the SAME architecture settings as in Phase 1.
def new_backbone(num_layers=None):
    if ARCH == 'a1':
        return DecoderTransformerLM(vocab_size=len(tokenizer),context_length=2048,
            d_model=640,n_heads=10,n_layers=num_layers or 8,d_ff=2560,dropout=0.1)
    if ARCH == 'a2':
        return EncoderDecoderTransformer(vocab_size=len(tokenizer),source_length=1536,
            target_length=512,d_model=600,n_heads=15,
            n_encoder_layers=num_layers or 8,n_decoder_layers=8,d_ff=2048,dropout=0.1)
    if ARCH == 'a3':
        return ModernLocalTransformerLM(vocab_size=len(tokenizer),
            dim=saved_cfg.get('dim',600),num_layers=num_layers or saved_cfg.get('num_layers',8),
            num_heads=saved_cfg.get('num_heads',10),num_kv_heads=saved_cfg.get('num_kv_heads',5),
            ffn_dim=saved_cfg.get('ffn_dim',1600),window_size=saved_cfg.get('window_size',512),
            chunk_size=saved_cfg.get('chunk_size',256),dropout=saved_cfg.get('dropout',0.1))
    if ARCH == 'a4':
        patterns=saved_cfg.get('attention_patterns',[
            {'type':'local','value':256}, {'type':'dilated','value':2},
            {'type':'local','value':512}, {'type':'dilated','value':4},
            {'type':'local','value':512}, {'type':'dilated','value':8},
            {'type':'local','value':1024}, {'type':'global','value':None}])
        return MultiScaleDilatedLM(vocab_size=len(tokenizer),
            context_length=saved_cfg.get('context_length',2048),
            dim=saved_cfg.get('dim',600),num_heads=saved_cfg.get('num_heads',10),
            ffn_dim=saved_cfg.get('ffn_dim',1600),attention_patterns=patterns,
            dropout=saved_cfg.get('dropout',0.1))
    raise ValueError('Unknown architecture: ' + ARCH)

if weight_file.endswith('.safetensors'):
    from safetensors.torch import load_file
    p1_state = load_file(str(weights_path))
else:
    p1_state = torch.load(weights_path, map_location='cpu', weights_only=True)

# Some Phase 1 notebooks saved a dictionary containing model_state.
if isinstance(p1_state, dict) and 'model_state' in p1_state:
    p1_state = p1_state['model_state']
if not isinstance(p1_state, dict):
    raise ValueError('Phase 1 checkpoint was not a state dictionary.')

# A1 uses output_layer.weight, not lm_head.weight.
# Do not add new keys to this state dict; loading with strict=True checks it.

# Architecture 3/4 tie the embedding and LM-head tensors in Phase 1.
# safetensors may save only one alias. Restore an alias ONLY when needed.
if ARCH in ('a3', 'a4'):
    left, right = 'token_embedding.weight', 'lm_head.weight'
    if left in p1_state and right not in p1_state:
        p1_state[right] = p1_state[left]
    if right in p1_state and left not in p1_state:
        p1_state[left] = p1_state[right]

base = new_backbone()
base.load_state_dict(p1_state, strict=True)
if ARCH == 'a1':
    base.output_layer = nn.Identity()  # NER uses hidden vectors, not word logits.
print('Loaded Phase 1 weights. Tokenizer size:', len(tokenizer))


Phase 1 repo: P-261110044/cs781-phase1-architecture3
Files in repo: ['.gitattributes', 'architecture_config.json', 'checkpoint-6000/model.safetensors', 'checkpoint-6000/optimizer.pt', 'checkpoint-6000/rng_state.pth', 'checkpoint-6000/scaler.pt', 'checkpoint-6000/scheduler.pt', 'checkpoint-6000/tokenizer.json', 'checkpoint-6000/tokenizer_config.json', 'checkpoint-6000/trainer_state.json', 'checkpoint-6000/training_args.bin', 'checkpoint-6104/model.safetensors']
Using weights: model_state.pt


model_state.pt:   0%|          | 0.00/247M [00:00<?, ?B/s]

architecture_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'TokenizersBackend'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


Loaded Phase 1 weights. Tokenizer size: 50000


In [4]:

LABELS=['APP','RESP','A.COUNSEL','R.COUNSEL','JUDGE','WIT',
        'AUTH','COURT','STAT','PREC','DATE','CASENO']
TAGS=['O']+[prefix+'-'+lab for lab in LABELS for prefix in ('B','I')]
TAG_ID={t:i for i,t in enumerate(TAGS)}
DIM=saved_cfg.get('dim',600)  # Match Phase 1 config

# LoRMA is multiplicative: apply small matrices to the OUTPUT of the frozen W.
# This is different from LoRA, which adds a low-rank term to W itself.
class LoRMALinear(nn.Module):
    def __init__(self,original):
        super().__init__()
        self.original=original
        self.original.requires_grad_(False)
        self.A=nn.Linear(original.out_features,RANK,bias=False)
        self.B=nn.Linear(RANK,original.out_features,bias=False)
        nn.init.zeros_(self.B.weight)
    def forward(self,x):
        y=self.original(x)
        return y+self.B(self.A(y))*(ALPHA/RANK)

# Normal residual adapter: shrink to a small dimension, then expand again.
class Adapter(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.down=nn.Linear(dim,64)
        self.up=nn.Linear(64,dim)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
    def forward(self,x):
        return self.up(F.gelu(self.down(x)))

class NERModel(nn.Module):
    def __init__(self,base,method):
        super().__init__()
        self.base=base
        self.classifier=nn.Linear(DIM,len(TAGS))
        self.dropout=nn.Dropout(0.1)
        if method not in ('full','kd'):
            self.base.requires_grad_(False)
        if method=='partial':
            # Only last 2 encoder blocks train; other weights remain frozen.
            for block in self.base.encoder_blocks[-2:]:block.requires_grad_(True)
        elif method=='adapters':
            self.adapters=nn.ModuleList([Adapter(DIM) for _ in self.base.blocks])
            for block,adapter in zip(self.base.blocks,self.adapters):
                block.register_forward_hook(lambda m,inp,out,ad=adapter: out+ad(out))
        elif method=='lorma':
            for block in self.base.blocks:
                block.attn.q_proj=LoRMALinear(block.attn.q_proj)
                block.attn.v_proj=LoRMALinear(block.attn.v_proj)
    def forward(self,input_ids):
        if ARCH=='a1':
            hidden=self.base(input_ids=input_ids).logits  # output_layer is Identity
        elif ARCH=='a2':
            hidden=self.base.encode(input_ids)
        else:
            hidden,_=self.base._run_hidden(input_ids)
        return self.classifier(self.dropout(hidden))

if TECHNIQUE=='kd':
    # A smaller student: keep the first 4 Phase 1 blocks (not random blocks).
    small=new_backbone(num_layers=4)
    small_state=small.state_dict()
    for key in small_state:
        if key not in p1_state or small_state[key].shape!=p1_state[key].shape:
            raise ValueError('Student weight was missing in Phase 1: '+key)
        small_state[key]=p1_state[key]
    small.load_state_dict(small_state,strict=True)
    base=small
    del small_state,small
    if ARCH=='a1':base.output_layer=nn.Identity()

del p1_state
model=NERModel(base,TECHNIQUE)

if TECHNIQUE=='lora':
    # PEFT is allowed by your TA. No TaskType here because this is OUR custom nn.Module,
    # not a Hugging Face AutoModelForTokenClassification.
    from peft import LoraConfig, get_peft_model
    peft_config=LoraConfig(r=RANK,lora_alpha=ALPHA,lora_dropout=0.05,
                           target_modules=['q_proj','v_proj'],
                           modules_to_save=['classifier'],bias='none')
    model=get_peft_model(model,peft_config)

model=model.to(DEVICE)
print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))
print('Total parameters:',sum(p.numel() for p in model.parameters()))


Trainable parameters: 149425
Total parameters: 61854650


In [5]:

# Official Phase-2 data only. Use fold1+fold2, then split by document ID.
# No test annotations are downloaded or used for validation.
def official_file(filename):
    return hf_hub_download(repo_id=DATA_REPO,
        filename='Phase_2_LNER/'+filename,repo_type='dataset',token=HF_TOKEN)
def read_official_json(path):
    with open(official_file(path),encoding='utf-8') as f: return json.load(f)

def extract_documents(obj):
    if isinstance(obj,dict):
        result={}
        for docid,v in obj.items():
            if isinstance(v,str): result[str(docid)]=v
            elif isinstance(v,dict):
                val=v.get('text',v.get('document',v.get('content')))
                if isinstance(val,list): val='\n'.join(val)
                if not isinstance(val,str): raise ValueError('Unexpected document text shape for '+str(docid))
                result[str(docid)]=val
            else: raise ValueError('Unexpected document value type for '+str(docid))
        return result
    if isinstance(obj,list):
        out={}
        for row in obj:
            docid=str(row.get('id',row.get('doc_id',row.get('document_id'))))
            val=row.get('text',row.get('document',row.get('content')))
            if isinstance(val,list): val='\n'.join(val)
            if not isinstance(val,str) or docid=='None':raise ValueError('Unexpected document row')
            if docid in out:raise ValueError('Duplicate document ID: '+docid)
            out[docid]=val
        return out
    raise ValueError('Unrecognized official text file format')

def one_entity(item):
    # Accept the official [start,end,label,text] form plus dictionary equivalents.
    if isinstance(item,(list,tuple)) and len(item)>=3:
        return (int(item[0]),int(item[1]),str(item[2]),str(item[3]) if len(item)>3 else None)
    if isinstance(item,dict):
        if 'span' in item and isinstance(item['span'],(list,tuple)):
            a,b=item['span'][:2]
        else:
            a=item.get('start',item.get('start_offset',item.get('start_char')))
            b=item.get('end',item.get('end_offset',item.get('end_char')))
        label=item.get('label',item.get('type',item.get('entity_type')))
        ent_text=item.get('text',item.get('entity_text'))
        if a is None or b is None or label is None:
            raise ValueError('Unrecognized annotation fields: '+str(item)[:250])
        return int(a),int(b),str(label),ent_text
    raise ValueError('Unrecognized entity format '+str(item)[:200])

def extract_annotations(gold):
    if isinstance(gold,list):
        gold={str(r.get('id',r.get('doc_id'))):r for r in gold}
    if not isinstance(gold,dict):raise ValueError('Unexpected gold JSON structure')
    out={}
    for docid, row in gold.items():
        if isinstance(row,dict):
            row=row.get('entities',row.get('annotations',row.get('spans',row.get('labels'))))
        if row is None: row=[]
        if not isinstance(row,list):raise ValueError('Gold annotations not a list for '+str(docid))
        out[str(docid)]=[one_entity(e) for e in row]
    return out

texts={}
golds={}
for fold in ('fold1','fold2'):
    tt=extract_documents(read_official_json('Train/'+fold+'-text.json'))
    gg=extract_annotations(read_official_json('Train/'+fold+'-gold.json'))
    assert tt.keys()==gg.keys(),f'Missing/extra IDs in {fold}: {len(tt)} texts, {len(gg)} golds'
    assert texts.keys().isdisjoint(tt), 'Duplicate IDs across training folds'
    texts.update(tt); golds.update(gg)

# Confirm label list agrees with the competition document (12 categories).
with open(official_file('Train/labels.txt'),encoding='utf-8') as f:
    official_labels=[line.strip() for line in f if line.strip()]
assert set(official_labels)==set(LABELS), ('Unexpected label names: ',official_labels)

warn_count=0
for docid, spans in golds.items():
    t=texts[docid]
    for a,b,lab,entity_text in spans:
        assert 0<=a<b<=len(t) and lab in LABELS,(docid,a,b,lab)
        if entity_text is not None and t[a:b]!=entity_text:
            # Some annotation sets may have mismatched auxiliary text; offsets/labels are authoritative.
            if warn_count<5: print('WARNING annotation text differs from text at offsets:',docid,repr(t[a:b]),repr(entity_text))
            warn_count+=1

ids=sorted(texts)
random.Random(SEED).shuffle(ids)
val_n=max(1,round(len(ids)*VAL_PART))
dev_ids=ids[:val_n]; train_ids=ids[val_n:]
if MAX_TRAIN_DOCS is not None:train_ids=train_ids[:MAX_TRAIN_DOCS]
if MAX_VAL_DOCS is not None:dev_ids=dev_ids[:MAX_VAL_DOCS]
assert set(train_ids).isdisjoint(dev_ids)
print('Train documents:',len(train_ids),'held-out documents:',len(dev_ids))
print('Training labels:',Counter(s[2] for did in train_ids for s in golds[did]))


fold1-text.json: 0.00B [00:00, ?B/s]

fold1-gold.json: 0.00B [00:00, ?B/s]

fold2-text.json: 0.00B [00:00, ?B/s]

fold2-gold.json: 0.00B [00:00, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Train documents: 63 held-out documents: 7
Training labels: Counter({'DATE': 1420, 'STAT': 1208, 'PREC': 1102, 'AUTH': 976, 'COURT': 832, 'CASENO': 697, 'APP': 390, 'RESP': 317, 'JUDGE': 203, 'WIT': 187, 'A.COUNSEL': 179, 'R.COUNSEL': 161})


In [6]:

# Byte/character offset safety: tokenize the ORIGINAL full document (no normalization).
# Every offset returned by the fast tokenizer points into the original source string.
# Replace the existing encode_document() function with this one.
# It uses the SAME tokenizer trained in Phase 1.
# The change is only in matching character spans to BPE tokens.

def encode_document(text, spans=None):
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
        return_attention_mask=False
    )
    input_ids = encoded['input_ids']
    offsets = [tuple(pair) for pair in encoded['offset_mapping']]
    tags = [TAG_ID['O']] * len(input_ids)

    if spans is not None:
        used_tokens = set()

        # Sort by start character. Longer overlapping entities go first.
        for a, b, label, _ in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):
            # Include BPE tokens that overlap the gold entity.
            # E.g. token ' S' has offsets (11,13), but gold starts at 12.
            token_idxs = [
                i for i, (start, end) in enumerate(offsets)
                if end > start and start < b and end > a
            ]
            if not token_idxs:
                continue

            # One token cannot represent two labels at the same time.
            if any(i in used_tokens for i in token_idxs):
                continue

            first_start = offsets[token_idxs[0]][0]
            last_end = offsets[token_idxs[-1]][1]

            # Crossing the gold boundary is okay if ONLY whitespace is outside.
            # If there are other letters/punctuation outside, strict offsets
            # cannot be recovered from a plain token BIO label.
            left_extra = text[first_start:a] if first_start < a else ''
            right_extra = text[b:last_end] if last_end > b else ''
            if left_extra.strip() or right_extra.strip():
                continue

            for k, i in enumerate(token_idxs):
                prefix = 'B-' if k == 0 else 'I-'
                tags[i] = TAG_ID[prefix + label]
                used_tokens.add(i)

    return input_ids, offsets, tags
def make_windows(input_ids,offsets,tag_ids=None):
    result=[]
    for begin in range(0,len(input_ids),STRIDE):
        end=min(begin+MAX_LEN,len(input_ids))
        if begin>=end:break
        window={'ids':input_ids[begin:end], 'offsets':offsets[begin:end], 'begin':begin}
        if tag_ids is not None:window['tags']=tag_ids[begin:end]
        result.append(window)
        if end==len(input_ids):break
    return result

class LNERWindows(Dataset):
    def __init__(self,doc_ids):
        self.examples=[]
        for did in doc_ids:
            a,b,c=encode_document(texts[did],golds[did])
            for row in make_windows(a,b,c):
                self.examples.append(row)
    def __len__(self):return len(self.examples)
    def __getitem__(self,i):return self.examples[i]

def collate_one(batch):
    assert len(batch)==1, 'Use train batch size 1; A2 encoder does not mask padding.'
    row=batch[0]
    return (torch.tensor([row['ids']],dtype=torch.long),
            torch.tensor([row['tags']],dtype=torch.long))

train_ds=LNERWindows(train_ids)
loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,
                  collate_fn=collate_one,num_workers=0)
print('Train windows:',len(train_ds))
assert len(train_ds)>0


Train windows: 2182


In [7]:
# Validation and prediction: pehle windows jodo, phir full BIO sequence decode karo.
from tqdm.auto import tqdm
import numpy as np

# Same offset conversion as before. For greedy tags, an invalid I tag starts a new entity.
def spans_from_tags(text, offsets, tag_ids, confidences=None):
    if confidences is None:
        confidences = [1.0] * len(tag_ids)
    spans = []
    current = None

    def close():
        nonlocal current
        if current is not None:
            a, b, label, conf = current
            # Original text ke start/end mein sirf extra spaces trim karo.
            piece = text[a:b]
            a += len(piece) - len(piece.lstrip())
            b -= len(piece) - len(piece.rstrip())
            if a < b:
                spans.append((a, b, label, sum(conf) / len(conf)))
            current = None

    for (s, e), tag_id, prob in zip(offsets, tag_ids, confidences):
        if not (0 <= s < e <= len(text)):
            close()
            continue
        tag = TAGS[int(tag_id)]
        if tag == 'O':
            close()
            continue
        prefix, label = tag.split('-', 1)
        if (current and prefix == 'I' and label == current[2]
                and not text[current[1]:s].strip()):
            current = (current[0], e, label, current[3] + [float(prob)])
        else:
            close()
            current = (s, e, label, [float(prob)])
    close()
    return spans


def prefer_longer_context(merged, best_left, start, window_logits):
    # Causal model mein token ko jis window se more LEFT context milta hai, wo lo.
    n = len(window_logits)
    context = torch.arange(n, dtype=torch.long)
    old = best_left[start:start+n]
    replace = context > old
    merged[start:start+n][replace] = window_logits[replace]
    old[replace] = context[replace]


def valid_bio_viterbi(logits):
    # Highest scoring BIO sequence dhundho, but invalid I transitions allow mat karo.
    # I-X ke just pehle B-X ya I-X hona chahiye.
    scores = logits.detach().float().cpu().numpy()
    length, n_tags = scores.shape
    if length == 0:
        return []

    b_ids = np.array([TAG_ID['B-' + lab] for lab in LABELS])
    i_ids = np.array([TAG_ID['I-' + lab] for lab in LABELS])
    parents = np.zeros((length, n_tags), dtype=np.int16)

    dp = scores[0].copy()
    dp[i_ids] = -1e9  # Sentence/document I-X se start nahi ho sakta

    for t in range(1, length):
        best_prev = int(dp.argmax())
        next_dp = scores[t] + dp[best_prev]
        parents[t, :] = best_prev

        # I-X can only continue B-X or I-X. B and O can follow any tag.
        take_b = dp[b_ids] >= dp[i_ids]
        valid_prev = np.where(take_b, b_ids, i_ids)
        next_dp[i_ids] = scores[t, i_ids] + dp[valid_prev]
        parents[t, i_ids] = valid_prev
        dp = next_dp

    # Backtrack from the last token to get final tag IDs.
    tags = [0] * length
    tags[-1] = int(dp.argmax())
    for t in range(length - 1, 0, -1):
        tags[t - 1] = int(parents[t, tags[t]])
    return tags


@torch.no_grad()
def predict_one_doc_legacy(document):
    # Purana method: each window ka span alag banao, confidence se merge karo.
    # Sirf comparison ke liye rakha hai. Final prediction neeche wala method hai.
    model.eval()
    ids, offsets, _ = encode_document(document, None)
    candidate = {}
    for window in make_windows(ids, offsets):
        inp = torch.tensor([window['ids']], dtype=torch.long, device=DEVICE)
        probabilities = model(inp)[0].softmax(dim=-1)
        confidence, tags = probabilities.max(dim=-1)
        spans = spans_from_tags(document, window['offsets'],
                                tags.cpu().tolist(), confidence.cpu().tolist())
        for a, b, label, conf in spans:
            key = (a, b, label)
            candidate[key] = max(conf, candidate.get(key, 0.0))
    ranked = sorted(candidate.items(), key=lambda row: (-row[1], -(row[0][1] - row[0][0])))
    chosen = []
    for (a, b, label), _ in ranked:
        if not any(a < end and start < b for start, end, _ in chosen):
            chosen.append((a, b, label))
    return sorted(chosen, key=lambda x: (x[0], x[1], x[2]))


@torch.no_grad()
def predict_one_doc(document, method=None):
    if method is None:
        method = DECODE_METHOD
    if method == 'legacy':
        return predict_one_doc_legacy(document)
    if method not in ('greedy', 'viterbi'):
        raise ValueError('Use legacy, greedy or viterbi')

    model.eval()
    ids, offsets, _ = encode_document(document, None)
    if not ids:
        return []

    # Full document ki logits yahan store kar rahe hain (CPU memory).
    merged = torch.empty((len(ids), len(TAGS)), dtype=torch.float32)
    best_left = torch.full((len(ids),), -1, dtype=torch.long)

    for window in make_windows(ids, offsets):
        inp = torch.tensor([window['ids']], dtype=torch.long, device=DEVICE)
        window_logits = model(inp)[0].detach().float().cpu()
        prefer_longer_context(merged, best_left, window['begin'], window_logits)

    assert (best_left >= 0).all(), 'Some document tokens were not covered by windows'

    if method == 'viterbi':
        tag_ids = valid_bio_viterbi(merged)
    else:
        tag_ids = merged.argmax(dim=-1).tolist()

    # Now decode exactly once using original full-document offsets.
    spans = spans_from_tags(document, offsets, tag_ids)
    return [(a, b, label) for a, b, label, _ in spans]


def strict_scores(document_ids, method=None):
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in tqdm(document_ids, desc='Validation', leave=False):
        predicted = set(predict_one_doc(texts[did], method=method))
        actual = {(a, b, label) for a, b, label, _ in golds[did]}
        for _, _, label in predicted & actual: tp[label] += 1
        for _, _, label in predicted - actual: fp[label] += 1
        for _, _, label in actual - predicted: fn[label] += 1

    scores, precisions, recalls = [], [], []
    for label in LABELS:
        precision = tp[label] / max(1, tp[label] + fp[label])
        recall = tp[label] / max(1, tp[label] + fn[label])
        f1 = 2 * precision * recall / max(1e-12, precision + recall)
        scores.append(f1)
        precisions.append(precision)
        recalls.append(recall)
    return {
        'strict_macro_f1': float(np.mean(scores)),
        'strict_macro_precision': float(np.mean(precisions)),
        'strict_macro_recall': float(np.mean(recalls)),
        'per_label_f1': dict(zip(LABELS, map(float, scores)))
    }

# Simple tests: no model or GPU needed.
assert spans_from_tags('The Delhi High Court sat.', [(4,9), (10,14), (15,20)],
       [TAG_ID['B-COURT'], TAG_ID['I-COURT'], TAG_ID['I-COURT']]) == [(4,20,'COURT',1.0)]
assert spans_from_tags('ABC', [(0,1)], [TAG_ID['O']]) == []
assert max(len(w['ids']) for w in make_windows(list(range(400)), [(0,1)]*400)) <= MAX_LEN

# First token ko I-COURT ka high score do. Viterbi should still return valid BIO.
toy = torch.full((2, len(TAGS)), -5.0)
toy[0, TAG_ID['I-COURT']] = 8.0
toy[0, TAG_ID['B-COURT']] = 7.0
toy[1, TAG_ID['I-COURT']] = 8.0
assert [TAGS[i] for i in valid_bio_viterbi(toy)] == ['B-COURT', 'I-COURT']

# Overlap mein later window ko prefer karna chahiye, because more left context.
toy_merge = torch.empty((4, len(TAGS)))
toy_left = torch.full((4,), -1, dtype=torch.long)
prefer_longer_context(toy_merge, toy_left, 0, torch.ones((3, len(TAGS))))
prefer_longer_context(toy_merge, toy_left, 1, torch.full((3, len(TAGS)), 2.0))
assert toy_merge[:, 0].tolist() == [1.0, 1.0, 1.0, 2.0]
print('Stitching and valid-BIO unit tests passed')


Stitching and valid-BIO unit tests passed


In [8]:
# Check token/entity alignment BEFORE training.
# This is important: in an earlier A3 run only 4 of 139 gold spans
# could be reconstructed. More epochs cannot fix bad offsets.
# These checks use the TRUE labels; no GPU or model predictions are needed.
from collections import Counter

def check_alignment(doc_ids, show_examples=True):
    total_gold = 0
    total_recovered = 0
    mismatched_text = 0
    tag_counts = Counter()
    missing_examples = []

    for docid in doc_ids:
        doc = texts[docid]
        gold_spans = golds[docid]
        ids, offsets, tags = encode_document(doc, gold_spans)
        tag_counts.update(TAGS[tag] for tag in tags)
        correct = {(a, b, lab) for a, b, lab, _ in gold_spans}
        rebuilt = {(a, b, lab) for a, b, lab, _ in spans_from_tags(doc, offsets, tags)}
        total_gold += len(correct)
        total_recovered += len(correct & rebuilt)

        for a, b, lab, entity_text in gold_spans:
            if entity_text is not None and doc[a:b] != entity_text:
                mismatched_text += 1
        if show_examples:
            for a, b, lab in sorted(correct - rebuilt):
                if len(missing_examples) < 5:
                    missing_examples.append((lab, (a, b), repr(doc[a:b])[:90]))

    coverage = total_recovered / max(1, total_gold)
    print('Documents checked:', len(doc_ids))
    print('Gold spans:', total_gold)
    print('Gold spans reconstructed from TRUE BIO tags:', total_recovered)
    print('Exact gold-span coverage:', f'{coverage:.2%}')
    print('Annotation text mismatches:', mismatched_text)
    print('Most common BIO labels:', tag_counts.most_common(8))
    if show_examples:
        print('Examples missing after tokenization:', missing_examples)
    if coverage < 0.8:
        print('WARNING: Serious token/character alignment problem.')
        print('Do not assume that extra epochs or a longer MAX_LEN will fix this.')
    return coverage

# Find out WHY an annotation is missing from the BIO labels.
# This is a short summary; it cannot by itself correct boundary problems.
def count_skip_reasons(doc_ids):
    reasons = Counter()
    samples = []
    for did in doc_ids:
        doc = texts[did]
        _, offsets, _ = encode_document(doc, golds[did])
        occupied = set()
        for a, b, lab, _ in sorted(golds[did], key=lambda x: (x[0], -(x[1]-x[0]))):
            touching = [i for i, (s, e) in enumerate(offsets)
                        if e > s and s < b and e > a]
            if not touching:
                why = 'no_token_touching_span'
            elif occupied.intersection(touching):
                why = 'overlapping_annotations'
            else:
                first = offsets[touching[0]][0]
                last = offsets[touching[-1]][1]
                left = doc[first:a] if first < a else ''
                right = doc[b:last] if last > b else ''
                if left.strip() or right.strip():
                    why = 'non_whitespace_crosses_boundary'
                else:
                    why = 'accepted_by_encoder'
                    occupied.update(touching)
            reasons[why] += 1
            if why != 'accepted_by_encoder' and len(samples) < 5:
                samples.append((why, lab, repr(doc[a:b])[:65]))
    print('Why annotations are kept/skipped:', dict(reasons))
    print('Skipped entity examples:', samples)

# Check at least a few documents, not only the first validation document.
val_alignment = check_alignment(dev_ids[:min(5, len(dev_ids))])
train_alignment = check_alignment(train_ids[:min(5, len(train_ids))], show_examples=False)
count_skip_reasons(dev_ids[:min(5, len(dev_ids))])

# Set this to True if you want to stop instead of training on labels
# that cannot be recovered exactly. Leave False only for debugging.
STOP_IF_ALIGNMENT_BAD = False       # CHANGE to True before a serious run
if STOP_IF_ALIGNMENT_BAD and min(val_alignment, train_alignment) < 0.8:
    raise ValueError('Bad gold-span alignment. Fix preprocessing before full training.')


# A second pre-training test: entity spans should also survive the sliding windows.
# This is separate from the full-document tokenizer alignment check above.
def check_window_alignment(doc_ids):
    n_gold = 0
    n_found = 0
    missed_examples = []
    for did in doc_ids:
        doc = texts[did]
        ids, offsets, gold_tags = encode_document(doc, golds[did])
        target = {(a,b,lab) for a,b,lab,_ in golds[did]}
        decoded = set()
        for window in make_windows(ids, offsets, gold_tags):
            for a,b,lab,_ in spans_from_tags(doc, window['offsets'], window['tags']):
                decoded.add((a,b,lab))
        n_gold += len(target)
        n_found += len(target & decoded)
        for a,b,lab in sorted(target-decoded):
            if len(missed_examples) < 5:
                missed_examples.append((lab,(a,b),repr(doc[a:b])[:70]))
    print('Gold spans surviving sliding-window BIO decoding:', n_found, '/', n_gold)
    print('Window-level exact coverage:', f'{n_found / max(1,n_gold):.2%}')
    print('Window-level examples missed:', missed_examples)
    if n_gold and n_found/n_gold < 0.9:
        print('WARNING: window splitting is losing many spans. Review STRIDE and MAX_LEN.')

check_window_alignment(dev_ids[:min(5,len(dev_ids))])


Documents checked: 5
Gold spans: 488
Gold spans reconstructed from TRUE BIO tags: 483
Exact gold-span coverage: 98.98%
Annotation text mismatches: 0
Most common BIO labels: [('O', 21702), ('I-DATE', 558), ('I-CASENO', 388), ('I-PREC', 208), ('I-COURT', 204), ('B-DATE', 139), ('B-COURT', 135), ('I-STAT', 85)]
Examples missing after tokenization: [('AUTH', (1963, 1976), "'Railway Board'"), ('AUTH', (11218, 11231), "'Railway Board'"), ('PREC', (12849, 12901), "'Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]'"), ('PREC', (14121, 14174), "'Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]'"), ('COURT', (19254, 19264), "'High Court'")]
Documents checked: 5
Gold spans: 702
Gold spans reconstructed from TRUE BIO tags: 694
Exact gold-span coverage: 98.86%
Annotation text mismatches: 0
Most common BIO labels: [('O', 30652), ('I-PREC', 705), ('I-DATE', 512), ('I-STAT', 185), ('I-CASENO', 176), ('B-DATE', 127), ('I-AUTH', 117), ('I-COURT', 115)]
Why annotations are kept/skipped: {'accept

In [9]:
# Class imbalance hai, but O ko pehle jitna low weight nahi de rahe.
weights = torch.ones(len(TAGS), device=DEVICE)
weights[TAG_ID['O']] = O_WEIGHT

# Classifier naya hai, LoRA pretrained model ke andar add hua hai.
# Dono ko separate LR dena simple rahega.
head_params, lora_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if 'classifier' in name:
        head_params.append(param)
    else:
        lora_params.append(param)
assert head_params and lora_params, 'Expected both classifier and LoRA trainable weights'
optimizer = torch.optim.AdamW([
    {'params': lora_params, 'lr': LR_LORA},
    {'params': head_params, 'lr': LR_HEAD},
], weight_decay=0.01)
print('Trainable LoRA params:', sum(p.numel() for p in lora_params))
print('Trainable classifier params:', sum(p.numel() for p in head_params))
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

teacher=None
if TECHNIQUE=='kd':
    assert ARCH=='a3'
    teacher_base=new_backbone() # original full-size Phase-1 Architecture 3
    teacher=NERModel(teacher_base,'lora')
    # The trained teacher used PEFT LoRA, so recreate the same PEFT wrapper.
    from peft import LoraConfig,get_peft_model
    teacher=get_peft_model(teacher,LoraConfig(r=RANK,lora_alpha=ALPHA,
                         lora_dropout=0.05,target_modules=['q_proj','v_proj'],
                         modules_to_save=['classifier'],bias='none'))
    teacher_path = Path(TEACHER_FILE)
    if not teacher_path.is_file() and TEACHER_REPO_NAME:
        # Download the teacher if A3 LoRA trained in a different Kaggle session.
        teacher_repo = HF_USER + '/' + TEACHER_REPO_NAME
        print('Downloading teacher from:', teacher_repo)
        teacher_path = Path(hf_hub_download(repo_id=teacher_repo,
                            filename='a3_lora_best.pt',repo_type='model',token=HF_TOKEN))
    if not teacher_path.is_file():
        raise FileNotFoundError('A3 LoRA teacher checkpoint missing. Train A3 LoRA first.')
    teacher_weights=torch.load(teacher_path,map_location='cpu',weights_only=True)
    teacher.load_state_dict(teacher_weights['model_state'],strict=True)
    teacher=teacher.to(DEVICE).eval()
    teacher.requires_grad_(False)
    assert sum(p.numel() for p in model.parameters()) < sum(p.numel() for p in teacher.parameters())
    print('Teacher and student sizes:',sum(p.numel() for p in teacher.parameters()),
          sum(p.numel() for p in model.parameters()))

# Displaying time here does not change how the model trains.
def format_time(seconds):
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

best_f1=-1.0
best_epoch=0
no_improve=0  # Last best score ke baad kitne epochs improve nahi hue
history=[]
CKPT=OUTPUT / (EXP_ID+'_best.pt')
all_start=time.perf_counter()

for epoch in range(1,EPOCHS+1):
    epoch_start=time.perf_counter()
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total=0.0

    # tqdm shows completed batches, speed and approximate time remaining.
    progress=tqdm(enumerate(loader),total=len(loader),
                  desc=f'Epoch {epoch}/{EPOCHS} training')

    for i,(input_ids,tags) in progress:
        input_ids=input_ids.to(DEVICE)
        tags=tags.to(DEVICE)

        with torch.autocast(device_type=DEVICE.type,enabled=(DEVICE.type=='cuda')):
            logits=model(input_ids)
            loss=F.cross_entropy(logits.reshape(-1,len(TAGS)),
                                 tags.reshape(-1),weight=weights)
            if teacher is not None:
                with torch.no_grad():
                    teach_logits=teacher(input_ids)
                distill=F.kl_div(F.log_softmax(logits/KD_TEMP,dim=-1),
                                 F.softmax(teach_logits/KD_TEMP,dim=-1),
                                 reduction='batchmean')
                distill=distill/input_ids.size(1)
                loss=(1-KD_MIX)*loss+KD_MIX*(KD_TEMP**2)*distill
            loss=loss/ACCUM_STEPS

        scaler.scale(loss).backward()

        if (i+1)%ACCUM_STEPS==0 or i+1==len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total+=float(loss.detach())*ACCUM_STEPS

        # Only refresh the loss display sometimes, to keep it lightweight.
        if (i+1)%10==0 or i+1==len(loader):
            progress.set_postfix(loss=f'{total/(i+1):.4f}')

    train_seconds=time.perf_counter()-epoch_start

    # Validation has a separate progress bar.
    val_start=time.perf_counter()
    metrics=strict_scores(dev_ids)
    val_seconds=time.perf_counter()-val_start
    epoch_seconds=time.perf_counter()-epoch_start

    metrics.update({
        'epoch':epoch,
        'mean_training_loss':total/len(loader),
        'train_seconds':round(train_seconds,2),
        'validation_seconds':round(val_seconds,2),
        'epoch_seconds':round(epoch_seconds,2)
    })
    history.append(metrics)

    print(f"Epoch {epoch}/{EPOCHS} | loss {metrics['mean_training_loss']:.4f} | "
          f"strict macro-F1 {metrics['strict_macro_f1']:.4f}")
    print('Train time:',format_time(train_seconds),
          '| Validation time:',format_time(val_seconds),
          '| Total epoch time:',format_time(epoch_seconds))

    with open(OUTPUT/'history.json','w') as f:
        json.dump(history,f,indent=2)

    if metrics['strict_macro_f1']>best_f1:
        # Better val F1 mila toh best model save karo aur patience reset karo.
        best_f1=metrics['strict_macro_f1']
        best_epoch=epoch
        no_improve=0
        torch.save({'model_state':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                    'epoch':epoch,'labels':TAGS,'technique':TECHNIQUE,'architecture':ARCH,
                    'phase1_repo':HF_REPO,'metrics':metrics,
                    'max_len':MAX_LEN,'stride':STRIDE,'o_weight':O_WEIGHT,
                    'lr_lora':LR_LORA,'lr_head':LR_HEAD,'decode_method':DECODE_METHOD},CKPT)
        print('Saved best checkpoint:',CKPT)
    else:
        no_improve+=1
        print(f'No F1 improvement: {no_improve}/{PATIENCE} epochs')
        if no_improve>=PATIENCE:
            print(f'Early stopping at epoch {epoch}. Best epoch was {best_epoch}.')
            break

print('Total training + validation time:',format_time(time.perf_counter()-all_start))
print(f'Best held-out macro-F1: {best_f1:.4f} (epoch {best_epoch})')

# Last epoch zaroori nahi ki best ho, isliye saved best weights load kar rahe hain.
best_state = torch.load(CKPT, map_location='cpu', weights_only=True)
model.load_state_dict(best_state['model_state'], strict=True)
model.eval()
print('Best model loaded from epoch:', best_state['epoch'])
del best_state


Trainable LoRA params: 134400
Trainable classifier params: 15025


Epoch 1/30 training:   0%|          | 0/2182 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 1/30 | loss 0.4423 | strict macro-F1 0.2501
Train time: 00:01:39 | Validation time: 00:00:03 | Total epoch time: 00:01:42
Saved best checkpoint: /kaggle/working/phase2_a3_lora_v3_30ep/a3_lora_v3_30ep_best.pt


Epoch 2/30 training:   0%|          | 0/2182 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

... (51 outputs left out)

Epoch 19/30 | loss 0.0879 | strict macro-F1 0.3928
Train time: 00:01:38 | Validation time: 00:00:03 | Total epoch time: 00:01:41
No F1 improvement: 4/5 epochs


Epoch 20/30 training:   0%|          | 0/2182 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 20/30 | loss 0.0846 | strict macro-F1 0.3942
Train time: 00:01:38 | Validation time: 00:00:03 | Total epoch time: 00:01:41
No F1 improvement: 5/5 epochs
Early stopping at epoch 20. Best epoch was 15.
Total training + validation time: 00:34:07
Best held-out macro-F1: 0.4043 (epoch 15)
Best model loaded from epoch: 15


In [10]:
# Optional extra checks after training. Does the model predict mostly O?
# This only prints numbers; it does not change the trained model.
from collections import Counter

print('Gold BIO tags across training windows:')
actual_counts = Counter()
for row in train_ds.examples:
    actual_counts.update(TAGS[t] for t in row['tags'])
print(actual_counts.most_common(15))
print('Entity-token fraction:',
      round(sum(v for t, v in actual_counts.items() if t != 'O') /
            max(1, sum(actual_counts.values())), 5))

# Check a few windows that actually contain gold entities.
examples_with_entities = [row for row in train_ds.examples
                          if any(t != TAG_ID['O'] for t in row['tags'])][:5]
predicted_counts = Counter()
model.eval()
with torch.no_grad():
    for row in examples_with_entities:
        sample = torch.tensor([row['ids']], dtype=torch.long, device=DEVICE)
        output = model(sample)[0].argmax(-1).tolist()
        predicted_counts.update(TAGS[t] for t in output)
print('Predicted BIO tags on 5 entity-containing windows:')
print(predicted_counts.most_common(15))

# Repeat the gold round-trip test on the validation document.
check_alignment(dev_ids[:1])


Gold BIO tags across training windows:
[('O', 506122), ('I-PREC', 12087), ('I-DATE', 7301), ('I-CASENO', 6484), ('I-STAT', 4122), ('B-DATE', 1864), ('I-AUTH', 1595), ('B-STAT', 1574), ('I-COURT', 1462), ('B-PREC', 1386), ('B-AUTH', 1301), ('I-APP', 1076), ('B-COURT', 1066), ('B-CASENO', 882), ('I-RESP', 847)]
Entity-token fraction: 0.08519
Predicted BIO tags on 5 entity-containing windows:
[('O', 1046), ('I-PREC', 121), ('I-DATE', 24), ('I-JUDGE', 20), ('I-STAT', 19), ('I-COURT', 7), ('B-DATE', 6), ('I-APP', 5), ('B-COURT', 5), ('B-PREC', 5), ('I-RESP', 4), ('B-JUDGE', 4), ('I-AUTH', 4), ('B-STAT', 4), ('B-APP', 2)]
Documents checked: 1
Gold spans: 139
Gold spans reconstructed from TRUE BIO tags: 139
Exact gold-span coverage: 100.00%
Annotation text mismatches: 0
Most common BIO labels: [('O', 4306), ('I-DATE', 175), ('I-CASENO', 95), ('B-DATE', 44), ('I-JUDGE', 37), ('B-RESP', 36), ('I-APP', 34), ('I-RESP', 34)]
Examples missing after tokenization: []


1.0

In [11]:
# Optional Hugging Face upload for THIS Phase 2 experiment.
# A 2-document quick test should not overwrite a real model.
UPLOAD_TO_HF = True               # CHANGE to True after proper training
PHASE2_REPO_NAME = 'phase2-a3-lora'  # CHANGE this repo name if you want

if UPLOAD_TO_HF:
    phase2_repo = HF_USER + '/' + PHASE2_REPO_NAME
    api.create_repo(repo_id=phase2_repo, repo_type='model',
                    private=True, exist_ok=True)
    # Save the trained checkpoint and training log.
    for path in (CKPT, OUTPUT / 'history.json'):
        if path.is_file():
            api.upload_file(path_or_fileobj=str(path), path_in_repo=path.name,
                            repo_id=phase2_repo, repo_type='model')
    # Keep the tokenizer and config used in Phase 1 with the model.
    for name in ('architecture_config.json', 'tokenizer.json',
                 'tokenizer_config.json', 'special_tokens_map.json'):
        src = P1_DIR / name
        if src.is_file():
            api.upload_file(path_or_fileobj=str(src), path_in_repo=name,
                            repo_id=phase2_repo, repo_type='model')
    print('Uploaded Phase 2 checkpoint to:', phase2_repo)
else:
    print('HF upload skipped. Set UPLOAD_TO_HF=True after proper training.')


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded Phase 2 checkpoint to: P-261110044/phase2-a3-lora


In [12]:

# Reload the selected BEST Phase-2 checkpoint, then predict official fold3.
# Exact Codabench output: {doc_id: [[start_char,end_char,entity_label], ...]}.
checkpoint=torch.load(CKPT,map_location='cpu',weights_only=True)
assert checkpoint['labels']==TAGS and checkpoint['technique']==TECHNIQUE
model.load_state_dict(checkpoint['model_state'],strict=True)
model.eval()

# Same trained weights, just compare greedy vs valid BIO decoding on validation.
legacy_metrics = strict_scores(dev_ids, method='legacy')
greedy_metrics = strict_scores(dev_ids, method='greedy')
viterbi_metrics = strict_scores(dev_ids, method='viterbi')
print('Validation F1 - old window method:', round(legacy_metrics['strict_macro_f1'], 4))
print('Validation F1 - stitched greedy:', round(greedy_metrics['strict_macro_f1'], 4))
print('Validation F1 - stitched Viterbi:', round(viterbi_metrics['strict_macro_f1'], 4))
decoder_f1 = {
    'legacy': legacy_metrics['strict_macro_f1'],
    'greedy': greedy_metrics['strict_macro_f1'],
    'viterbi': viterbi_metrics['strict_macro_f1'],
}
if AUTO_SELECT_DECODER:
    # Test ke labels unknown hain, isliye method sirf validation se choose karte hain.
    DECODE_METHOD = max(decoder_f1, key=decoder_f1.get)
print('Submission decoding:', DECODE_METHOD)
with open(OUTPUT/'decoder_comparison.json', 'w') as f:
    json.dump({'scores': decoder_f1, 'selected': DECODE_METHOD}, f, indent=2)

test_texts=extract_documents(read_official_json('Test/fold3-text.json'))
assert len(test_texts)>0
predictions={}
for i,(docid,doc_text) in enumerate(test_texts.items()):
    predictions[docid]=[[int(a),int(b),lab] for a,b,lab in predict_one_doc(doc_text)]
    if (i+1)%20==0:print('Predicted',i+1,'/',len(test_texts))

assert set(predictions)==set(test_texts)
for did,entities in predictions.items():
    for item in entities:
        assert len(item)==3 and isinstance(item[0],int) and isinstance(item[1],int)
        assert 0<=item[0]<item[1]<=len(test_texts[did]) and item[2] in LABELS

pred_path=OUTPUT/'prediction.json'
with open(pred_path,'w',encoding='utf-8') as f:
    json.dump(predictions,f,ensure_ascii=False,indent=2)
zip_path=OUTPUT/'submission.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path,arcname='prediction.json')
with zipfile.ZipFile(zip_path) as zf:
    assert zf.namelist()==['prediction.json']
    verified=json.loads(zf.read('prediction.json'))
    assert set(verified)==set(test_texts)
print('Ready for Codabench:',zip_path)
print('Total test documents:',len(predictions),'predicted spans:',sum(map(len,predictions.values())))


Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Validation:   0%|          | 0/7 [00:00<?, ?it/s]

Validation F1 - old window method: 0.2745
Validation F1 - stitched greedy: 0.2844
Validation F1 - stitched Viterbi: 0.4043
Submission decoding: viterbi


fold3-text.json: 0.00B [00:00, ?B/s]

Predicted 20 / 35
Ready for Codabench: /kaggle/working/phase2_a3_lora_v3_30ep/submission.zip
Total test documents: 35 predicted spans: 4734


# Just Checking different things to improve macro-f1 score

In [13]:
from collections import Counter
import pandas as pd

def to_span_set(items):
    # Gold/prediction dono ko (start, end, label) format mein convert karo
    if isinstance(items, dict):
        for key in ('entities', 'annotations', 'spans'):
            if key in items:
                items = items[key]
                break

    result = set()

    for item in items:
        if isinstance(item, dict):
            start = item.get('start', item.get('start_char'))
            end = item.get('end', item.get('end_char'))
            label = item.get('label')
        else:
            start, end, label = item[:3]

        result.add((int(start), int(end), str(label)))

    return result


def diagnose_spans(name, doc_ids, max_docs=None):
    gold_counts = Counter()
    pred_counts = Counter()
    correct_counts = Counter()

    missed = []
    extra = []

    ids = list(doc_ids)
    if max_docs is not None:
        ids = ids[:max_docs]

    for doc_id in ids:
        doc_id = str(doc_id)
        text = texts[doc_id]

        gold = to_span_set(golds[doc_id])
        pred = to_span_set(predict_one_doc(text))

        # Exact match: start, end aur label teeno same hone chahiye
        correct = gold & pred

        gold_counts.update(s[2] for s in gold)
        pred_counts.update(s[2] for s in pred)
        correct_counts.update(s[2] for s in correct)

        missed.extend((doc_id, s) for s in gold - pred)
        extra.extend((doc_id, s) for s in pred - gold)

    rows = []

    for label in LABELS:
        g = gold_counts[label]
        p = pred_counts[label]
        tp = correct_counts[label]

        precision = tp / p if p else 0.0
        recall = tp / g if g else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        rows.append([label, g, p, tp, precision, recall, f1])

    df = pd.DataFrame(
        rows,
        columns=[
            'Label', 'Gold', 'Predicted', 'Correct',
            'Precision', 'Recall', 'F1'
        ]
    )

    macro_f1 = df['F1'].mean()

    print(f"\n===== {name} =====")
    print("Documents:", len(ids))
    print("Gold spans:", sum(gold_counts.values()))
    print("Predicted spans:", sum(pred_counts.values()))
    print("Exact matches:", sum(correct_counts.values()))
    print(f"Strict Macro-F1: {macro_f1:.4f}")

    print("\nPredicted entity distribution:")
    print(pred_counts.most_common())

    print("\nPer-class performance:")
    print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    print("\nMissed gold entities (first 5):")
    for doc_id, (start, end, label) in missed[:5]:
        print(label, repr(texts[doc_id][start:end]))

    print("\nExtra/wrong predicted entities (first 5):")
    for doc_id, (start, end, label) in extra[:5]:
        print(label, repr(texts[doc_id][start:end]))

    return macro_f1, df


# 1. Evaluate the held-out validation documents
val_f1, val_details = diagnose_spans(
    "VALIDATION",
    dev_ids
)

# 2. Evaluate documents ACTUALLY used during training
# Assumes training used the first MAX_TRAIN_DOCS IDs in train_ids.
# If training IDs were randomly sampled, replace this with that exact list.
if 'train_ids' in globals():
    n_used = globals().get('MAX_TRAIN_DOCS', len(train_ids))
    actual_train_ids = list(train_ids)[:n_used] if n_used is not None else list(train_ids)

    train_f1, train_details = diagnose_spans(
        "TRAIN SAMPLE",
        actual_train_ids,
        max_docs=20
    )

    print("\n===== TRAIN VS VALIDATION =====")
    print(f"Train sample F1: {train_f1:.4f}")
    print(f"Validation F1:   {val_f1:.4f}")
else:
    print("Set actual_train_ids to the document IDs used for training.")
    print("Then run diagnose_spans('TRAIN SAMPLE', actual_train_ids, max_docs=20)")


===== VALIDATION =====
Documents: 7
Gold spans: 721
Predicted spans: 875
Exact matches: 447
Strict Macro-F1: 0.4043

Predicted entity distribution:
[('COURT', 229), ('DATE', 176), ('PREC', 101), ('CASENO', 92), ('STAT', 75), ('AUTH', 73), ('JUDGE', 36), ('APP', 27), ('R.COUNSEL', 25), ('A.COUNSEL', 19), ('WIT', 15), ('RESP', 7)]

Per-class performance:
    Label  Gold  Predicted  Correct  Precision  Recall    F1
      APP    17         27        3      0.111   0.176 0.136
     RESP    64          7        2      0.286   0.031 0.056
A.COUNSEL    16         19       10      0.526   0.625 0.571
R.COUNSEL    23         25       15      0.600   0.652 0.625
    JUDGE    42         36       12      0.333   0.286 0.308
      WIT     0         15        0      0.000   0.000 0.000
     AUTH    27         73       14      0.192   0.519 0.280
    COURT   181        229      152      0.664   0.840 0.741
     STAT    52         75       32      0.427   0.615 0.504
     PREC    81        101       2

In [14]:
sample = train_ds[0]

print("Sample type:", type(sample))

if isinstance(sample, dict):
    print("Available keys:", list(sample.keys()))
    for k, v in sample.items():
        print(k, ":", getattr(v, "shape", type(v)))
else:
    print("Sample:", str(sample)[:500])

Sample type: <class 'dict'>
Available keys: ['ids', 'offsets', 'begin', 'tags']
ids : <class 'list'>
offsets : <class 'list'>
begin : <class 'int'>
tags : <class 'list'>


In [15]:
import torch
from collections import Counter

@torch.inference_mode()
def diagnose_bio(model, dataset, n_windows=100):

    pred_counts = Counter()
    gold_counts = Counter()

    device = next(model.parameters()).device
    was_training = model.training
    model.eval()

    try:
        # Sample windows from the dataset
        step = max(1, len(dataset) // n_windows)
        indices = list(range(0, len(dataset), step))[:n_windows]

        for i in indices:
            batch = dataset[i]

            # Your dataset uses 'ids' and 'tags'
            ids = torch.tensor(
                batch['ids'], dtype=torch.long,
                device=device
            ).unsqueeze(0)

            gold_tags = batch['tags']

            # Forward pass
            output = model(ids)

            if hasattr(output, 'logits'):
                output = output.logits
            elif isinstance(output, dict):
                output = output['logits']
            elif isinstance(output, tuple):
                output = output[0]

            # Shape: [sequence_length, 25]
            if output.ndim == 3:
                output = output[0]

            predicted = output.argmax(dim=-1).cpu().tolist()

            if len(predicted) != len(gold_tags):
                raise ValueError("Prediction and gold lengths differ")

            for pred_id, gold_id in zip(predicted, gold_tags):

                # Ignore tokens without valid gold labels
                if gold_id is None or gold_id == -100:
                    continue

                gold_name = (
                    gold_id if isinstance(gold_id, str)
                    else TAGS[int(gold_id)]
                )

                pred_counts[TAGS[int(pred_id)]] += 1
                gold_counts[gold_name] += 1

    finally:
        model.train(was_training)

    total = sum(pred_counts.values())

    print("\n===== PREDICTED BIO TAGS =====")
    for tag, count in pred_counts.most_common():
        print(f"{tag:15s} {count}")

    print("\n===== GOLD BIO TAGS =====")
    for tag, count in gold_counts.most_common():
        print(f"{tag:15s} {count}")

    print("\n===== SUMMARY =====")

    print("Windows checked:", len(indices))
    print("Total evaluated tokens:", total)

    if total > 0:
        print(f"Predicted O: {100 * pred_counts['O'] / total:.2f}%")
        print(f"Gold O: {100 * gold_counts['O'] / total:.2f}%")

    print("Predicted B tags:",
          sum(v for k, v in pred_counts.items()
              if k.startswith('B-')))

    print("Gold B tags:",
          sum(v for k, v in gold_counts.items()
              if k.startswith('B-')))

    print("Predicted I tags:",
          sum(v for k, v in pred_counts.items()
              if k.startswith('I-')))

    print("Gold I tags:",
          sum(v for k, v in gold_counts.items()
              if k.startswith('I-')))


# Run diagnostics on 100 training windows
diagnose_bio(model, train_ds, n_windows=100)


===== PREDICTED BIO TAGS =====
O               22949
I-PREC          681
I-DATE          322
I-CASENO        320
I-STAT          225
I-COURT         100
B-AUTH          99
I-AUTH          96
B-DATE          90
B-STAT          78
B-PREC          69
I-APP           61
B-COURT         60
I-RESP          50
I-JUDGE         43
B-CASENO        42
I-R.COUNSEL     28
I-WIT           26
I-A.COUNSEL     19
B-APP           18
B-RESP          18
B-R.COUNSEL     14
B-WIT           13
B-JUDGE         11
B-A.COUNSEL     11

===== GOLD BIO TAGS =====
O               23152
I-PREC          641
I-DATE          326
I-CASENO        281
I-STAT          175
B-DATE          85
I-COURT         79
B-AUTH          75
I-AUTH          73
B-PREC          71
I-APP           69
B-STAT          66
B-COURT         55
I-JUDGE         53
I-RESP          47
B-CASENO        36
I-WIT           29
B-APP           23
I-R.COUNSEL     23
I-A.COUNSEL     19
B-RESP          17
B-WIT           16
B-JUDGE         11
B-A.COUNSEL   

In [16]:
from collections import Counter
import torch

@torch.inference_mode()
def check_invalid_bio(model, dataset, n_windows=100):

    device = next(model.parameters()).device
    was_training = model.training
    model.eval()

    invalid_pred = Counter()
    invalid_gold = Counter()

    total_pred_I = 0
    total_gold_I = 0

    indices = list(range(
        0, len(dataset),
        max(1, len(dataset) // n_windows)
    ))[:n_windows]

    try:
        for i in indices:
            sample = dataset[i]

            ids = torch.tensor(
                sample['ids'],
                dtype=torch.long,
                device=device
            ).unsqueeze(0)

            gold_ids = sample['tags']

            output = model(ids)

            if hasattr(output, 'logits'):
                output = output.logits
            elif isinstance(output, dict):
                output = output['logits']
            elif isinstance(output, tuple):
                output = output[0]

            if output.ndim == 3:
                output = output[0]

            pred_ids = output.argmax(-1).cpu().tolist()

            prev_pred = 'O'
            prev_gold = 'O'

            for p, g in zip(pred_ids, gold_ids):

                if g is None or g == -100:
                    prev_pred = 'O'
                    prev_gold = 'O'
                    continue

                pred = TAGS[int(p)]
                gold = g if isinstance(g, str) else TAGS[int(g)]

                if pred.startswith('I-'):
                    total_pred_I += 1
                    label = pred[2:]

                    if prev_pred not in ('B-' + label, 'I-' + label):
                        invalid_pred[label] += 1

                if gold.startswith('I-'):
                    total_gold_I += 1
                    label = gold[2:]

                    if prev_gold not in ('B-' + label, 'I-' + label):
                        invalid_gold[label] += 1

                prev_pred = pred
                prev_gold = gold

    finally:
        model.train(was_training)

    print("===== INVALID BIO TRANSITIONS =====")

    print("\nPredicted I tags:", total_pred_I)
    print("Invalid predicted I tags:", sum(invalid_pred.values()))

    if total_pred_I:
        print("Invalid percentage:",
              f"{100 * sum(invalid_pred.values()) / total_pred_I:.2f}%")

    print("\nInvalid predicted tags by label:")
    print(invalid_pred.most_common())

    print("\nInvalid gold tags by label:")
    print(invalid_gold.most_common())

    print("\nNote: Window starts and ignored positions can create")
    print("invalid transitions even with correct original annotations.")


check_invalid_bio(model, train_ds, n_windows=100)

===== INVALID BIO TRANSITIONS =====

Predicted I tags: 1971
Invalid predicted I tags: 133
Invalid percentage: 6.75%

Invalid predicted tags by label:
[('PREC', 42), ('CASENO', 19), ('AUTH', 18), ('COURT', 9), ('APP', 8), ('STAT', 7), ('WIT', 7), ('RESP', 6), ('DATE', 6), ('JUDGE', 4), ('R.COUNSEL', 4), ('A.COUNSEL', 3)]

Invalid gold tags by label:
[('CASENO', 3), ('PREC', 3), ('COURT', 1), ('APP', 1), ('STAT', 1), ('DATE', 1)]

Note: Window starts and ignored positions can create
invalid transitions even with correct original annotations.
